Step 1: Preprocessing meteostat data

In [83]:
from meteostat import Point, Hourly
from datetime import datetime
import pandas as pd
import numpy as np

Set location and time param

In [84]:
# Parameters
lat, lon = 34.0008, -81.0351
start = datetime(2020,1,1)
end = datetime(2025,1,1)
city = Point(lat, lon)

Get data

In [104]:
# Fetch data from api
df = Hourly(city, start, end).fetch()
#print(df)

# ensure timestamps are datetime
df.index = pd.to_datetime(df.index)

# Select & rename columns
df = df[['temp', 'dwpt', 'rhum', 'prcp', 'wdir', 'wspd', 'pres', 'coco']]

# Ensure hourly continuity
full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H', tz=df.index.tz)
df = df.reindex(full_idx)

Fill in precips with NaN = 0 and log transform, add boolean rain flag

In [103]:
# assume NaN prcp = 0
df['prcp'] = df['prcp'].fillna(0)
# Change precip to either 0 or log-transformed amount
df['prcp'] = np.log1p(df['prcp'])
# add a rain flag (0 or 1)
df['prcp_flag'] = (df['prcp'] > 0).astype(int)
#print(df)

If timestamps missing (unlikely) fill them in by interpolating small gaps and forward fill long gaps

In [96]:
# Impute missing timestamps
# short gaps (<=3h): linear; long gaps: forward fill and add mask
gap_mask = df.isna().any(axis=1)
df_short = df.interpolate(limit=3, limit_direction='both')
df_long = df_short.fillna(method='ffill')


Standardize variables and clip outliers

In [102]:
# Standardize numerical units and clip outliers (3 sigma)
df_std= df_long.copy()
for col in ['temp', 'dwpt', 'rhum', 'wspd', 'pres']:
    var = df_long[col]
    mu, sigma = var.mean(), var.std()
    # clip values greater than 3 stds
    clipped = var.clip(lower=mu - 3*sigma, upper=mu + 3*sigma)

#print(df_std)

Encoding time using cyclical sin/cos transformation 

In [97]:
# Convert time to seconds
timestamp_s = df_std.index.map(pd.Timestamp.timestamp)
print(timestamp_s)

# Get the number of seconds for each time period
day = 24*60*60
week = day*7
year = day*(365.2425)

# Transform using sin and cos
# Time of day
df_std['day_sin'] = np.sin(timestamp_s * (2 * np.pi / day))
df_std['day_cos'] = np.cos(timestamp_s * (2 * np.pi / day))

# Time of week
df_std['week_sin'] = np.sin(timestamp_s * (2 * np.pi / week))
df_std['week_cos'] = np.cos(timestamp_s * (2 * np.pi / week))

# Time of year
df_std['year_sin'] = np.sin(timestamp_s * (2 * np.pi / year))
df_std['year_cos'] = np.cos(timestamp_s * (2 * np.pi / year))



Index([1577923200.0, 1577926800.0, 1577930400.0, 1577934000.0, 1577937600.0,
       1577941200.0, 1577944800.0, 1577948400.0, 1577952000.0, 1577955600.0,
       ...
       1735657200.0, 1735660800.0, 1735664400.0, 1735668000.0, 1735671600.0,
       1735675200.0, 1735678800.0, 1735682400.0, 1735686000.0, 1735689600.0],
      dtype='float64', length=43825)


Encode wind speed as sin/cos too since it is circular

In [98]:
df_std['wdir_sin'] = np.sin(np.deg2rad(df_std['wdir']))
df_std['wdir_cos'] = np.cos(np.deg2rad(df_std['wdir']))


See which features are most correlated to determine while lag variables to create
Don't want to create redundant lag variabels

In [91]:
df_std[['temp', 'rhum', 'dwpt', 'wspd', 'wdir', 'pres']].corr()

,temp,rhum,dwpt,wspd,wdir,pres
temp,1.000000,-0.199511,0.791236,0.215625,0.204937,-0.398432
rhum,-0.199511,1.000000,0.430254,-0.430961,-0.369017,-0.094538
dwpt,0.791236,0.430254,1.000000,-0.065698,-0.038680,-0.416926
wspd,0.215625,-0.430961,-0.065698,1.000000,0.714394,-0.272037
wdir,0.204937,-0.369017,-0.038680,0.714394,1.000000,-0.291609
pres,-0.398432,-0.094538,-0.416926,-0.272037,-0.291609,1.000000


Adding lag features (temp, pressure, humidity, wind speed)

In [99]:

# more important variables, longer lag
lag_hours = [1, 3, 6, 24]
vars_to_lag = ['temp', 'rhum', 'pres']

for var in vars_to_lag:
    for lag in lag_hours:
        df_std[f'{var}_lag{lag}'] = df_std[var].shift(lag)

# less important, shorter lag to reduce noise
# dew point correlates with 
lag_hours = [1, 3, 6]
vars_to_lag = ['wspd', 'prcp', 'dwpt', 'wdir_sin', 'wdir_cos']

for var in vars_to_lag:
    for lag in lag_hours:
        df_std[f'{var}_lag{lag}'] = df_std[var].shift(lag)

#print(df_std)



Rolling stats

In [100]:
# rolling average
roll_hours = [3, 6, 12]
vars_to_roll = ['temp', 'rhum', 'wspd', 'prcp', 'dwpt']

for var in vars_to_roll:
    for window in roll_hours:
        df_std[f'{var}_roll{window}'] = df_std[var].shift(window).rolling(window, min_periods=window).mean()

# do sum of precip instead of avg
for window in [3, 6, 12]:
    df_std[f'prcp_sum{window}'] = df_std['prcp'].shift(1).rolling(window, min_periods=1).sum()
#print(df_std)

Drop NA's resulting from lag and rolling

In [101]:
# drop NA's resulting from lag
lag_roll_cols = [col for col in df_std.columns if ('lag' in col) or ('roll' in col) or ('sum' in col)]

df_std = df_std.dropna(subset=lag_roll_cols).copy()
#print(df_std)

In [95]:
# Save cleaned dataset
df_std.to_parquet('hourly_columbia_weather.parquet')